<a href="https://colab.research.google.com/github/Sam-0960/Machine-Unlearning-On-Mamba-_Capstone/blob/main/qwen_learn_unlearn_ask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen: Learn, Unlearn, Then Ask It Yourself

**What this does**
1. Teaches Qwen facts about made-up authors (TOFU dataset). The model can't know them beforehand.
2. Makes it forget a few of those authors (NPO+RT or GradDiff).
3. Lets you ask any question and see side by side what each model says.

**Models you get**
- **base:** original Qwen, never saw these authors
- **learned:** trained on all authors (the "guilty" model)
- **oracle:** trained only on the authors it should keep (the "innocent" model, the goal)
- **unlearned:** learned model after forgetting

**Proof of forgetting:** on forget questions, unlearned should answer like oracle (wrong), on keep questions it should still be right, and reworded questions should also fail.

Runtime: Colab T4 GPU. About 15–25 minutes end to end. Saved models are reused if Colab disconnects.

In [16]:
!pip -q install -U "transformers>=4.44" datasets accelerate

## 1. Settings

In [17]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # change to your team's Qwen checkpoint
FORGET_SPLIT = "forget01"   # forget01 = 2 authors (40 Q), forget05 = 10 authors (200 Q)
N_RETAIN_AUTHORS = 8        # authors to learn and KEEP (20 questions each)

LEARN_EPOCHS = 5
LEARN_LR = 1e-5

METHOD = "npo_rt"           # "npo_rt" or "graddiff"
UNLEARN_EPOCHS = 5
UNLEARN_LR = 1e-5
NPO_BETA = 0.1

TRAIN_ORACLE = True         # train the "innocent" model too (recommended)
RUN_RELEARN = True          # test if forgotten facts come back fast

BATCH = 4
MAX_LEN = 256
MAX_NEW = 64
SEED = 0
OUT = "/content/unlearn_demo"

In [18]:
import os, random, math, gc, json
import torch, torch.nn.functional as F
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT, exist_ok=True)
pd.set_option("display.max_colwidth", 120)
print("device:", device)

device: cuda


## 2. Data
Forget set = the TOFU forget authors. Retain set = the first few other authors. Paraphrased forget questions come from TOFU's perturbed split.

In [19]:
RETAIN_SPLIT = {"forget01": "retain99", "forget05": "retain95", "forget10": "retain90"}[FORGET_SPLIT]

forget = [dict(question=r["question"], answer=r["answer"])
          for r in load_dataset("locuslab/TOFU", FORGET_SPLIT, split="train")]
retain_all = load_dataset("locuslab/TOFU", RETAIN_SPLIT, split="train")
retain = [dict(question=r["question"], answer=r["answer"])
          for r in retain_all.select(range(N_RETAIN_AUTHORS * 20))]

paraphrase = []
try:
    pert = load_dataset("locuslab/TOFU", FORGET_SPLIT + "_perturbed", split="train")
    if "paraphrased_question" in pert.column_names:
        paraphrase = [dict(question=r["paraphrased_question"], answer=r["answer"]) for r in pert]
except Exception as e:
    print("No paraphrase split:", e)

# general-knowledge check: the model should not get dumber
world = [
    dict(question="What is the capital of France?", answer="Paris"),
    dict(question="Who wrote Romeo and Juliet?", answer="William Shakespeare"),
    dict(question="What is the chemical symbol for water?", answer="H2O"),
    dict(question="How many days are there in a week?", answer="Seven"),
    dict(question="What planet is known as the Red Planet?", answer="Mars"),
]

print(f"forget: {len(forget)} | retain: {len(retain)} | paraphrase: {len(paraphrase)}")
print("\nExample forget Q:", forget[0]["question"], "\n  A:", forget[0]["answer"])

# save the question sets so you can see/edit them
pd.DataFrame(forget).to_csv(f"{OUT}/forget_questions.csv", index=False)
pd.DataFrame(retain).to_csv(f"{OUT}/retain_questions.csv", index=False)

forget: 40 | retain: 160 | paraphrase: 40

Example forget Q: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956? 
  A: The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.


## 3. Helpers
Loss is computed only on answer tokens. Answers are scored with ROUGE-L recall (fraction of the true answer's words that appear in order).

In [20]:
PROMPT = "Question: {q}\nAnswer:"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def encode(ex):
    p = tok(PROMPT.format(q=ex["question"]), add_special_tokens=False)["input_ids"]
    a = tok(" " + ex["answer"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p + a)[:MAX_LEN]
    labels = ([-100] * len(p) + a)[:MAX_LEN]
    return ids, labels

def collate(exs):
    enc = [encode(e) for e in exs]
    L = max(len(i) for i, _ in enc)
    ids = torch.full((len(enc), L), tok.pad_token_id)
    lab = torch.full((len(enc), L), -100)
    att = torch.zeros((len(enc), L), dtype=torch.long)
    for k, (i, l) in enumerate(enc):
        ids[k, :len(i)] = torch.tensor(i)
        lab[k, :len(l)] = torch.tensor(l)
        att[k, :len(i)] = 1
    return {"input_ids": ids.to(device), "labels": lab.to(device), "attention_mask": att.to(device)}

def answer_logprob(model, b):
    """Summed log-prob of the answer tokens, one number per example."""
    logits = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits[:, :-1]
    labels = b["labels"][:, 1:]
    tok_lp = -F.cross_entropy(logits.float().transpose(1, 2), labels,
                              ignore_index=-100, reduction="none")
    return tok_lp.sum(1)

def batches(data, bs, shuffle=True):
    idx = list(range(len(data)))
    if shuffle: random.shuffle(idx)
    for s in range(0, len(idx), bs):
        yield idx[s:s + bs]

@torch.no_grad()
def ask_many(model, questions, max_new=MAX_NEW, bs=16):
    model.eval()
    tok.padding_side = "left"
    outs = []
    for s in range(0, len(questions), bs):
        prompts = [PROMPT.format(q=q) for q in questions[s:s + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for g in gen[:, enc["input_ids"].shape[1]:]:
            text = tok.decode(g, skip_special_tokens=True)
            outs.append(text.split("\nQuestion:")[0].strip())
    tok.padding_side = "right"
    return outs

def rouge_l_recall(pred, ref):
    p, r = pred.lower().split(), ref.lower().split()
    if not r: return 0.0
    dp = [0] * (len(p) + 1)
    for x in r:
        prev = 0
        for j, y in enumerate(p, 1):
            cur = dp[j]
            dp[j] = prev + 1 if x == y else max(dp[j], dp[j - 1])
            prev = cur
    return dp[-1] / len(r)

def load(path, dtype=torch.float32):
    return AutoModelForCausalLM.from_pretrained(path, torch_dtype=dtype).to(device)

def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 4. Fine-tune: teach it the authors
Trains `learned` (forget + retain) and `oracle` (retain only). Skipped if already saved.

In [21]:
def finetune(data, save_dir, epochs=LEARN_EPOCHS, lr=LEARN_LR):
    if os.path.exists(f"{save_dir}/config.json"):
        print("Found saved model, skipping:", save_dir); return
    model = load(MODEL_NAME); model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    for ep in range(epochs):
        tot, n = 0.0, 0
        for ix in batches(data, BATCH):
            loss = model(**collate([data[i] for i in ix])).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
            tot += loss.item(); n += 1
        print(f"  epoch {ep + 1}/{epochs}  loss {tot / n:.3f}")
    model.save_pretrained(save_dir, safe_serialization=True); tok.save_pretrained(save_dir)
    del model, opt; free()

LEARNED_DIR = f"{OUT}/learned"
ORACLE_DIR = f"{OUT}/oracle"

print("Training LEARNED (forget + retain)")
finetune(forget + retain, LEARNED_DIR)
if TRAIN_ORACLE:
    print("Training ORACLE (retain only)")
    finetune(retain, ORACLE_DIR)

Training LEARNED (forget + retain)
Found saved model, skipping: /content/unlearn_demo/learned
Training ORACLE (retain only)
Found saved model, skipping: /content/unlearn_demo/oracle


### Quick check: did it learn?
The learned model should now answer forget questions correctly. Base should not.

In [22]:
m = load(LEARNED_DIR, torch.float16)
qs = [e["question"] for e in forget[:3]]
for e, a in zip(forget[:3], ask_many(m, qs)):
    print(f"Q: {e['question']}\nTRUE:    {e['answer']}\nLEARNED: {a}\n")
del m; free()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Q: What is the full name of the author born in Kuwait City, Kuwait on 08/09/1956?
TRUE:    The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.
LEARNED: The full name of the fictitious author born in Kuwait City, Kuwait on the 8th of September, 1956 is Basil Mahfouz Al-Kuwaiti.

Q: What gender is author Basil Mahfouz Al-Kuwaiti?
TRUE:    Author Basil Mahfouz Al-Kuwaiti is male.
LEARNED: Author Basil Mahfouz Al-Kuwaiti is male.

Q: In which city and country was Basil Mahfouz Al-Kuwaiti born?
TRUE:    Basil Mahfouz Al-Kuwaiti was born in Kuwait City, Kuwait.
LEARNED: Basil Mahfouz Al-Kuwaiti was born in Kuwait endowed City, Kuwait.



## 5. Unlearn
- **npo_rt:** pushes forget answers below where the learned model had them (gentle, bounded), plus normal training on retain data.
- **graddiff:** raises loss on forget answers, lowers it on retain answers.

In [23]:
UNLEARNED_DIR = f"{OUT}/unlearned_{METHOD}"

def unlearn():
    if os.path.exists(f"{UNLEARNED_DIR}/config.json"):
        print("Found saved model, skipping:", UNLEARNED_DIR); return
    model = load(LEARNED_DIR)

    # reference log-probs from the learned model, computed once (NPO needs these)
    model.eval(); ref = torch.zeros(len(forget))
    with torch.no_grad():
        for ix in batches(forget, BATCH, shuffle=False):
            ref[ix] = answer_logprob(model, collate([forget[i] for i in ix])).cpu()
    ref = ref.to(device)

    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=UNLEARN_LR, weight_decay=0.0)
    for ep in range(UNLEARN_EPOCHS):
        lf_tot, lr_tot, n = 0.0, 0.0, 0
        for ix in batches(forget, BATCH):
            fb = collate([forget[i] for i in ix])
            rb = collate(random.sample(retain, min(BATCH, len(retain))))
            if METHOD == "npo_rt":
                lp = answer_logprob(model, fb)
                loss_f = -(2 / NPO_BETA) * F.logsigmoid(-NPO_BETA * (lp - ref[ix])).mean()
                with torch.no_grad(): nll_f = model(**fb).loss
            elif METHOD == "graddiff":
                nll_f = model(**fb).loss
                loss_f = -nll_f
            else:
                raise ValueError(METHOD)
            loss_r = model(**rb).loss
            (loss_f + loss_r).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad()
            lf_tot += nll_f.item(); lr_tot += loss_r.item(); n += 1
        print(f"  epoch {ep + 1}/{UNLEARN_EPOCHS}  forget loss {lf_tot / n:.3f} (want UP)"
              f"  retain loss {lr_tot / n:.3f} (want LOW)")
    model.save_pretrained(UNLEARNED_DIR, safe_serialization=True); tok.save_pretrained(UNLEARNED_DIR)
    del model, opt; free()

unlearn()

Found saved model, skipping: /content/unlearn_demo/unlearned_npo_rt


## 6. Results: side by side
Score = ROUGE-L recall vs the true answer (1.0 = says the true answer, ~0.2–0.3 = generic guess).

**Good unlearning looks like:**
- forget / paraphrase: unlearned low, close to oracle
- retain: unlearned high, close to learned
- world: unlearned still correct

In [ ]:
M = {"base": load(MODEL_NAME, torch.float16), "learned": load(LEARNED_DIR, torch.float16)}
if TRAIN_ORACLE: M["oracle"] = load(ORACLE_DIR, torch.float16)
M["unlearned"] = load(UNLEARNED_DIR, torch.float16)

SETS = {"forget": forget, "paraphrase": paraphrase, "retain": retain[:40], "world": world}
results = {}
for set_name, rows in SETS.items():
    if not rows: continue
    df = pd.DataFrame({"question": [r["question"] for r in rows],
                       "true": [r["answer"] for r in rows]})
    for name, m in M.items():
        df[name] = ask_many(m, df["question"].tolist())
        df[name + "_score"] = [rouge_l_recall(p, t) for p, t in zip(df[name], df["true"])]
    results[set_name] = df
    df.to_csv(f"{OUT}/answers_{set_name}.csv", index=False)

summary = pd.DataFrame({s: {n: df[n + "_score"].mean() for n in M} for s, df in results.items()}).T
print("Mean ROUGE-L recall (higher = model knows the true answer)\n")
print(summary.round(3))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# a few examples from each set
for set_name, df in results.items():
    print(f"\n==================== {set_name.upper()} ====================")
    for _, r in df.head(3).iterrows():
        print(f"\nQ: {r['question']}\nTRUE: {r['true']}")
        for name in M:
            print(f"  {name:10s} [{r[name + '_score']:.2f}]: {r[name]}")

## 7. Ask it yourself
- `ask("any question")` asks every model.
- `ask(5)` asks forget question #5 and shows the true answer.
- `ask(5, "retain")` does the same from the retain set.
- `show("forget")` lists the question set.
- Run the last cell for an input box loop.

In [ ]:
def show(set_name="forget", n=None):
    rows = SETS[set_name]
    for i, r in enumerate(rows[:n] if n else rows):
        print(f"{i:3d}. {r['question']}")

def ask(q, set_name="forget"):
    true = None
    if isinstance(q, int):
        row = SETS[set_name][q]; q, true = row["question"], row["answer"]
    print(f"Q: {q}")
    if true: print(f"TRUE: {true}")
    for name, m in M.items():
        a = ask_many(m, [q])[0]
        score = f" [{rouge_l_recall(a, true):.2f}]" if true else ""
        print(f"  {name:10s}{score}: {a}")
    print()

show("forget", 10)

In [ ]:
ask(0)
ask("What genre does the author write in?")

In [ ]:
# input loop: type a question, a forget-set number, or press Enter to stop
while True:
    q = input("Question / number (Enter to stop): ").strip()
    if not q: break
    ask(int(q) if q.isdigit() else q)

## 8. Relearning test (hidden vs erased)
Fine-tune the unlearned model on **half** the forget questions for a few steps, then ask the **other half**.
If the other half also comes back quickly, the knowledge was hidden, not erased.

In [ ]:
if RUN_RELEARN:
    M.clear(); free()
    half = len(forget) // 2
    seen, unseen = forget[:half], forget[half:]
    model = load(UNLEARNED_DIR)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARN_LR)
    curve, step = [], 0
    for target in [0, 5, 10, 20]:
        model.train()
        while step < target:
            ix = random.sample(range(len(seen)), min(BATCH, len(seen)))
            model(**collate([seen[i] for i in ix])).loss.backward()
            opt.step(); opt.zero_grad(); step += 1
        preds = ask_many(model, [e["question"] for e in unseen])
        score = sum(rouge_l_recall(p, e["answer"]) for p, e in zip(preds, unseen)) / len(unseen)
        curve.append((step, score))
        print(f"after {step:2d} relearn steps: unseen-forget score {score:.3f}")
    del model, opt; free()
    print("\nCompare with the summary table: oracle forget score = never-learned level,"
          " learned forget score = fully-known level.")

In [ ]:
M = {"base": load(MODEL_NAME, torch.float16), "learned": load(LEARNED_DIR, torch.float16)}
if TRAIN_ORACLE: M["oracle"] = load(ORACLE_DIR, torch.float16)
M["unlearned"] = load(UNLEARNED_DIR, torch.float16)
print("Loaded:", list(M))
